In [ ]:
#Librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import zipfile
import io


In [ ]:
# Fase 1 Extraccion

# Vamos a extraer los datos de viajes de Ecobici de un mes específico.
# Fuente: Portal de Datos Abiertos de la CDMX.
# Usaremos los datos de Julio de 2026 como ejemplo.
url = "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/08/public_data_web_2026-07.csv"

csv_file_name = "2026-07.csv"
print(f"Descargando datos desde: {url}")
try:
    response = requests.get(url, timeout=1200)  # Added a timeout of 600 seconds (10 minutes)
    response.raise_for_status()  # Raise an exception for bad status codes
    print("Descarga completada con éxito.")

except requests.exceptions.Timeout as e:
    print(f"Error de timeout durante la descarga: {e}")
    df_raw = pd.DataFrame()
except requests.exceptions.RequestException as e:
    print(f"Error durante la descarga: {e}")
    df_raw = pd.DataFrame()

Descargando datos desde: https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/08/public_data_web_2026-07.csv
Descarga completada con éxito.


In [ ]:
# Save the zip file locally
with open(csv_file_name, 'wb') as f:
    f.write(response.content)
print(f"Archivo CSV guardado como {csv_file_name}")

# Read the extracted CSV into a DataFrame
print(f"Leyendo datos desde: {csv_file_name}")
df_raw = pd.read_csv(csv_file_name)
print("Extracción completada con éxito.")
print(f"Se cargaron {df_raw.shape[0]} registros.")

Archivo CSV guardado como 2026-07.csv
Leyendo datos desde: 2026-07.csv
Extracción completada con éxito.
Se cargaron 1493484 registros.


In [ ]:
#Mostrar el tamaño del Dataframe (filas y columnas)
print("Tamaño del dataframe:")
print(df_raw.shape)

#Mostrar una previsualizacion de los datos
print(" Previsualisacion del dataframe: ")
display(df_raw.head(10))

Tamaño del dataframe:
(1493484, 9)
 Previsualisacion del dataframe: 


,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_EstacionArribo,Fecha_Arribo,Hora_Arribo
0,F,26.0,5552989,085,30/06/2026,23:43:41,503,01/07/2026,00:00:00
1,M,33.0,5128335,259,30/06/2026,23:52:28,266-267,01/07/2026,00:00:00
2,M,34.0,8647703,040,30/06/2026,23:41:57,011,01/07/2026,00:00:03
3,M,34.0,5633250,492,30/06/2026,23:56:52,489,01/07/2026,00:00:03
4,O,41.0,8516015,133,30/06/2026,23:31:01,345,01/07/2026,00:00:05
5,M,35.0,6215123,013,30/06/2026,23:21:18,244,01/07/2026,00:00:07
6,M,45.0,3861267,465,30/06/2026,23:54:41,476,01/07/2026,00:00:08
7,M,45.0,6013460,449,30/06/2026,23:22:36,682,01/07/2026,00:00:08
8,M,41.0,5971320,011,30/06/2026,23:28:39,503,01/07/2026,00:00:09
9,M,27.0,8516676,451,30/06/2026,23:01:17,005,01/07/2026,00:00:11


In [ ]:
# ==================================================
# FASE 2: TRANSFORMACIÓN (Transform)
# ==================================================
print("\n--- FASE 2: TRANSFORMACIÓN ---")

# Hacemos una copia para no modificar el DataFrame original
df = df_raw.copy()

# --- 2.1 Limpieza de Datos (Cleaning) ---
print("\nIniciando limpieza de datos...")
# Convertir columnas de fecha a formato datetime
# Esto es crucial para poder realizar cálculos y extraer componentes.
df['Fecha_Retiro'] = pd.to_datetime(df['Fecha_Retiro'], dayfirst=True)
df['Fecha_Arribo'] = pd.to_datetime(df['Fecha_Arribo'], dayfirst=True)
print("Columnas de fecha convertidas a datetime.")
display(df.head(10))


--- FASE 2: TRANSFORMACIÓN ---

Iniciando limpieza de datos...
Columnas de fecha convertidas a datetime.


,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_EstacionArribo,Fecha_Arribo,Hora_Arribo
0,F,26.0,5552989,085,2026-06-30,23:43:41,503,2026-07-01,00:00:00
1,M,33.0,5128335,259,2026-06-30,23:52:28,266-267,2026-07-01,00:00:00
2,M,34.0,8647703,040,2026-06-30,23:41:57,011,2026-07-01,00:00:03
3,M,34.0,5633250,492,2026-06-30,23:56:52,489,2026-07-01,00:00:03
4,O,41.0,8516015,133,2026-06-30,23:31:01,345,2026-07-01,00:00:05
5,M,35.0,6215123,013,2026-06-30,23:21:18,244,2026-07-01,00:00:07
6,M,45.0,3861267,465,2026-06-30,23:54:41,476,2026-07-01,00:00:08
7,M,45.0,6013460,449,2026-06-30,23:22:36,682,2026-07-01,00:00:08
8,M,41.0,5971320,011,2026-06-30,23:28:39,503,2026-07-01,00:00:09
9,M,27.0,8516676,451,2026-06-30,23:01:17,005,2026-07-01,00:00:11


In [ ]:
# --- 2.2 Feature Engineering ---
print("\nIniciando Feature Engineering...")

# Para calcular la duración real, necesitamos combinar la fecha con la hora específica
df['Fecha_Retiro_Completa'] = pd.to_datetime(df['Fecha_Retiro'].dt.date.astype(str) + ' ' + df['Hora_Retiro'])
df['Fecha_Arribo_Completa'] = pd.to_datetime(df['Fecha_Arribo'].dt.date.astype(str) + ' ' + df['Hora_Arribo'])

# 1. Duración del viaje en minutos (usando las columnas completas)
df['duracion_minutos'] = (df['Fecha_Arribo_Completa'] - df['Fecha_Retiro_Completa']).dt.total_seconds() / 60

# 2. Día de la semana (0=Lunes, 6=Domingo)
df['dia_semana'] = df['Fecha_Retiro'].dt.dayofweek

# 3. Hora del día
df['hora_inicio'] = df['Fecha_Retiro_Completa'].dt.hour

# 4. Categoría de día (Fin de semana vs. Entre semana)
df['tipo_dia'] = df['dia_semana'].apply(lambda x: 'Fin de Semana' if x >= 5 else 'Entre Semana')

print("Nuevas características creadas con precisión: 'duracion_minutos', 'dia_semana', 'hora_inicio', 'tipo_dia'.")
display(df[['Fecha_Retiro_Completa', 'Fecha_Arribo_Completa', 'duracion_minutos']].head())


Iniciando Feature Engineering...
Nuevas características creadas con precisión: 'duracion_minutos', 'dia_semana', 'hora_inicio', 'tipo_dia'.


,Fecha_Retiro_Completa,Fecha_Arribo_Completa,duracion_minutos
0,2026-06-30 23:43:41,2026-07-01 00:00:00,16.316667
1,2026-06-30 23:52:28,2026-07-01 00:00:00,7.533333
2,2026-06-30 23:41:57,2026-07-01 00:00:03,18.100000
3,2026-06-30 23:56:52,2026-07-01 00:00:03,3.183333
4,2026-06-30 23:31:01,2026-07-01 00:00:05,29.066667


In [ ]:
df.head(10)

,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_EstacionArribo,Fecha_Arribo,Hora_Arribo,duracion_minutos,dia_semana,hora_inicio,tipo_dia,duracion_normalizada,viaje_largo,Fecha_Retiro_Completa,Fecha_Arribo_Completa
0,F,26.0,5552989,085,2026-06-30,23:43:41,503,2026-07-01,00:00:00,16.316667,1,23,Entre Semana,0.000726,True,2026-06-30 23:43:41,2026-07-01 00:00:00
1,M,33.0,5128335,259,2026-06-30,23:52:28,266-267,2026-07-01,00:00:00,7.533333,1,23,Entre Semana,0.000726,True,2026-06-30 23:52:28,2026-07-01 00:00:00
2,M,34.0,8647703,040,2026-06-30,23:41:57,011,2026-07-01,00:00:03,18.100000,1,23,Entre Semana,0.000726,True,2026-06-30 23:41:57,2026-07-01 00:00:03
3,M,34.0,5633250,492,2026-06-30,23:56:52,489,2026-07-01,00:00:03,3.183333,1,23,Entre Semana,0.000726,True,2026-06-30 23:56:52,2026-07-01 00:00:03
4,O,41.0,8516015,133,2026-06-30,23:31:01,345,2026-07-01,00:00:05,29.066667,1,23,Entre Semana,0.000726,True,2026-06-30 23:31:01,2026-07-01 00:00:05
5,M,35.0,6215123,013,2026-06-30,23:21:18,244,2026-07-01,00:00:07,38.816667,1,23,Entre Semana,0.000726,True,2026-06-30 23:21:18,2026-07-01 00:00:07
6,M,45.0,3861267,465,2026-06-30,23:54:41,476,2026-07-01,00:00:08,5.450000,1,23,Entre Semana,0.000726,True,2026-06-30 23:54:41,2026-07-01 00:00:08
7,M,45.0,6013460,449,2026-06-30,23:22:36,682,2026-07-01,00:00:08,37.533333,1,23,Entre Semana,0.000726,True,2026-06-30 23:22:36,2026-07-01 00:00:08
8,M,41.0,5971320,011,2026-06-30,23:28:39,503,2026-07-01,00:00:09,31.500000,1,23,Entre Semana,0.000726,True,2026-06-30 23:28:39,2026-07-01 00:00:09
9,M,27.0,8516676,451,2026-06-30,23:01:17,005,2026-07-01,00:00:11,58.900000,1,23,Entre Semana,0.000726,True,2026-06-30 23:01:17,2026-07-01 00:00:11


In [ ]:
# --- 2.3 Normalización / Estandarización ---
# Vamos a normalizar la duración del viaje para que esté en una escala de 0 a 1.
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df['duracion_normalizada'] = scaler.fit_transform(df[['duracion_minutos']])
print("\n'duracion_minutos' normalizada a una escala de 0 a 1.")

# --- 2.4 Encoding de Variables Categóricas ---
# La columna 'tipo_dia' es categórica. La convertiremos a números usando One-Hot Encoding.
df_encoded = pd.get_dummies(df, columns=['tipo_dia'], prefix='dia')
print("Variable 'tipo_dia' codificada con One-Hot Encoding.")

# --- 2.5 Balanceo de Clases ---
# Imaginemos que queremos predecir si un viaje es "muy largo" (> 60 min).
df['viaje_largo'] = df['duracion_minutos'] > 60
print("\nEjemplo de Balanceo de Clases:")
print("Distribución de 'viaje_largo' antes del balanceo:")
print(df['viaje_largo'].value_counts())

# --- Verificación del DataFrame Transformado ---
print("\n--- Vista previa del DataFrame transformado ---")
print(df_encoded.head())


'duracion_minutos' normalizada a una escala de 0 a 1.
Variable 'tipo_dia' codificada con One-Hot Encoding.

Ejemplo de Balanceo de Clases:
Distribución de 'viaje_largo' antes del balanceo:
viaje_largo
False    1484173
True        9311
Name: count, dtype: int64

--- Vista previa del DataFrame transformado ---
  Genero_Usuario  Edad_Usuario     Bici Ciclo_Estacion_Retiro Fecha_Retiro  \
0              F          26.0  5552989                   085   2026-06-30   
1              M          33.0  5128335                   259   2026-06-30   
2              M          34.0  8647703                   040   2026-06-30   
3              M          34.0  5633250                   492   2026-06-30   
4              O          41.0  8516015                   133   2026-06-30   

  Hora_Retiro Ciclo_EstacionArribo Fecha_Arribo Hora_Arribo  duracion_minutos  \
0    23:43:41                  503   2026-07-01    00:00:00         16.316667   
1    23:52:28              266-267   2026-07-01    00:00:0